<a href="https://colab.research.google.com/github/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/blob/main/my%20training%20code/step1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import yaml
import pandas as pd
import re
import time
import urllib.request
import os

def build_regex_patterns(yaml_url):
    response = urllib.request.urlopen(yaml_url)
    config = yaml.safe_load(response)

    patterns = {}
    for cat_key, cat_data in config['categories'].items():
        if 'coarse_keywords' in cat_data and cat_data['coarse_keywords']:
            patterns[cat_key] = '|'.join(map(re.escape, cat_data['coarse_keywords']))
    return patterns

def load_ignore_keywords(url_or_path):
    response = urllib.request.urlopen(url_or_path)
    return [line.decode('utf-8').strip() for line in response if line.strip() and not line.decode('utf-8').startswith('#')]

def run_coarse_filter(input_url, output_csv, normal_output_csv, yaml_url, text_column, chunk_size=200000):
    print(f"🚀 開始粗篩: 讀取線上資料集 '{input_url}' ...")
    start_time = time.time()

    # 1. 載入 YAML
    patterns = build_regex_patterns(yaml_url)
    print(f"載入類別規則: {list(patterns.keys())}")

    # 2. 💡 修正點：在迴圈「外面」載入 Ignore List，且使用 Raw 網址
    IGNORE_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/ignore.txt"
    ignore_words = load_ignore_keywords(IGNORE_URL)
    ignore_pattern = '|'.join(map(re.escape, ignore_words))
    print(f"載入豁免清單: 包含 {len(ignore_words)} 個字詞")
    print("-" * 40)

    first_chunk = True
    total_candidates = 0
    total_normal = 0
    total_processed = 0

    for chunk in pd.read_csv(input_url, chunksize=chunk_size, low_memory=False):
        if text_column not in chunk.columns:
            text_column = 'name' if 'name' in chunk.columns else chunk.columns[0]

        chunk[text_column] = chunk[text_column].fillna("").astype(str)

        valid_format_mask = chunk[text_column].str.len() >= 2
        valid_format_mask &= chunk[text_column].str.contains(r'[\u4e00-\u9fa5]', regex=True)

        trigger_mask = pd.Series(False, index=chunk.index)
        for cat_key, pattern in patterns.items():
            chunk[f'matched_{cat_key}'] = chunk[text_column].str.contains(pattern, na=False)
            trigger_mask = trigger_mask | chunk[f'matched_{cat_key}']

        # 3. 套用豁免規則：命中惡意字，且沒有命中豁免詞
        hit_ignore = chunk[text_column].str.contains(ignore_pattern, na=False)
        trigger_mask = trigger_mask & (~hit_ignore)

        candidates = chunk[trigger_mask & valid_format_mask].copy()
        normal = chunk[(~trigger_mask) & valid_format_mask].sample(frac=0.001, random_state=42)

        mode = 'w' if first_chunk else 'a'
        header = first_chunk
        candidates.to_csv(output_csv, mode=mode, header=header, index=False)
        normal.to_csv(normal_output_csv, mode=mode, header=header, index=False)

        total_processed += len(chunk)
        total_candidates += len(candidates)
        total_normal += len(normal)
        first_chunk = False

        print(f"🔄 掃描進度: {total_processed:,} 筆 | 累積抓出: {total_candidates:,} 筆候選")

    print("-" * 40)
    print(f"✅ 粗篩完成！總耗時: {time.time() - start_time:.2f} 秒")
    print(f"🎯 最終候選池大小: {total_candidates:,} 筆 (已儲存至 {output_csv})")
    print(f"🎯 正常樣本數量: {total_normal:,} 筆")

if __name__ == "__main__":
    run_coarse_filter(
        input_url='https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/nodes_page_1hop.csv',
        output_csv='candidate_pool_sanitized.csv',
        normal_output_csv='normal_pool_sample.csv',
        yaml_url='https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml',
        text_column='name',
        chunk_size=250000
    )

🚀 開始粗篩: 讀取線上資料集 'https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/nodes_page_1hop.csv' ...
載入類別規則: ['loan', 'porn', 'gambling', 'scam_recruitment']
載入豁免清單: 包含 12 個字詞
----------------------------------------
🔄 掃描進度: 250,000 筆 | 累積抓出: 543 筆候選
🔄 掃描進度: 500,000 筆 | 累積抓出: 1,160 筆候選
🔄 掃描進度: 750,000 筆 | 累積抓出: 2,061 筆候選
🔄 掃描進度: 1,000,000 筆 | 累積抓出: 3,122 筆候選
🔄 掃描進度: 1,250,000 筆 | 累積抓出: 3,859 筆候選
🔄 掃描進度: 1,500,000 筆 | 累積抓出: 4,439 筆候選
🔄 掃描進度: 1,750,000 筆 | 累積抓出: 5,061 筆候選
🔄 掃描進度: 2,000,000 筆 | 累積抓出: 5,995 筆候選
🔄 掃描進度: 2,030,772 筆 | 累積抓出: 6,113 筆候選
----------------------------------------
✅ 粗篩完成！總耗時: 21.02 秒
🎯 最終候選池大小: 6,113 筆 (已儲存至 candidate_pool_sanitized.csv)
🎯 正常樣本數量: 1,564 筆
